# MPEC Economic Dispatch Optimizer

Este notebook ejecuta el optimizador MPEC para la extracción de escenarios extremos de despacho económico.

## Requisitos previos
Debes tener los siguientes archivos CSV en formato UTF-8 con separador `;`:
- `escenarios_historicos_completos.csv`
- `parametros_tecnologicos.csv`
- `potencia_instalada_agregada_tecnologia.csv`

## Step 1: Montar Google Drive

Ejecuta esta celda para conectar tu Google Drive a Colab.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("Google Drive montado exitosamente")

## Step 2: Clonar el repositorio o usar archivos locales

Elige una opción:

### Opción A: Clonar del repositorio de GitHub

In [ ]:
import os
os.chdir('/content')

# Clona el repositorio (reemplaza con tu URL si es necesario)
!git clone https://github.com/tu-usuario/tfg.git
os.chdir('/content/tfg')
print("Repositorio clonado")

### Opción B: Usar archivos desde Google Drive

Si prefieres usar tus archivos desde Google Drive, actualiza las rutas en la siguiente celda.

In [ ]:
import os

# Actualiza estas rutas según donde tengas tus archivos en Google Drive
DRIVE_PATH = '/content/drive/MyDrive/TFG'  # Cambia esto a tu ruta
COLAB_PATH = '/content/tfg_solver'

os.makedirs(COLAB_PATH, exist_ok=True)
os.chdir(COLAB_PATH)

# Crear directorios necesarios
os.makedirs('data', exist_ok=True)
os.makedirs('results', exist_ok=True)

print(f"Directorio de trabajo: {os.getcwd()}")
print(f"Contenido de Drive: {os.listdir(DRIVE_PATH)}")

## Step 3: Instalar dependencias

In [ ]:
!pip install gurobipy pandas
print("Dependencias instaladas")

## Step 4: Subir archivos de datos (si no usas Google Drive)

Ejecuta esta celda para subir los archivos CSV necesarios.

In [ ]:
from google.colab import files
import os

print("Sube los 3 archivos CSV requeridos:")
print("  1. escenarios_historicos_completos.csv")
print("  2. parametros_tecnologicos.csv")
print("  3. potencia_instalada_agregada_tecnologia.csv")

uploaded = files.upload()

# Mover archivos a la carpeta 'data'
for filename in uploaded.keys():
    os.rename(filename, f'data/{filename}')
    print(f"Archivo {filename} movido a data/")

## Step 5: Descargar el script del solver

Si no lo tienes, descarga el script del repositorio.

In [ ]:
import urllib.request

# Descargar el script (reemplaza con la URL raw de tu repositorio)
url = 'https://raw.githubusercontent.com/tu-usuario/tfg/main/solver_por_tecnologia.py'
urllib.request.urlretrieve(url, 'solver_por_tecnologia.py')

print("Script descargado correctamente")

## Step 6: Configurar licencia de Gurobi

Necesitas las credenciales de Gurobi Web License Server (WLS).
Si no tienes, puedes obtener una licencia académica en https://www.gurobi.com/academia/

In [ ]:
# Las credenciales de Gurobi se pueden configurar de dos formas:

# Opción 1: Configurar variables de entorno (recomendado para Colab)
import os

os.environ['GUROBI_WLSACCESSID'] = 'tu_access_id_aqui'
os.environ['GUROBI_WLSSECRET'] = 'tu_secret_aqui'
os.environ['GUROBI_LICENSEID'] = 'tu_license_id_aqui'

print("Variables de entorno configuradas")
print("Nota: Reemplaza los valores con tus credenciales de Gurobi WLS")

## Step 7: Ejecutar el optimizador

In [ ]:
from solver_por_tecnologia import resolver_mpec_tramos_max_min

# Ejecutar el optimizador
resolver_mpec_tramos_max_min(data_dir='data', output_dir='results')

## Step 8: Descargar resultados

In [ ]:
import os
import shutil
from google.colab import files

# Comprimir resultados
shutil.make_archive('resultados_mpec', 'zip', 'results')

# Descargar
files.download('resultados_mpec.zip')

print("Resultados comprimidos y listos para descargar")

## Información sobre resultados

Los resultados se guardan en dos carpetas:
- **results/Max/**: Escenarios con máximo poder de mercado
- **results/Min/**: Escenarios con mínimo poder de mercado

Cada archivo CSV contiene:
- `Estacion`: Estación del año
- `Hora`: Hora del día
- `Demanda_MWh`: Demanda total
- `Lider_Evaluado`: Generador líder evaluado
- `Pi_m_Endogeno_EUR`: Precio endógeno del mercado
- `Beneficio_Lider_EUR`: Beneficio del líder
- `Tipo_Escenario`: MAX_PODER_MERCADO o MIN_PODER_MERCADO
- `Tramo`, `Tecnologia`, `C_Oferta_EUR`, `Potencia_Casada_MW`: Detalles de la casación